In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

# Paths
drive_path = "/content/drive/MyDrive/ccpd26dark"
local_path = "/content/ccpd26dark"


# Copy dataset to local storage
if not os.path.exists(local_path):
    print("Copying dataset to Colab local storage...")
    shutil.copytree(drive_path, local_path)
    print("Dataset copied successfully.")
else:
    print("Dataset already exists in local storage.")

# Fix data.yaml paths
data_yaml_path = os.path.join(local_path, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_yaml = f.read()

data_yaml = data_yaml.replace("/content/drive/MyDrive/ccpd26dark", local_path)

with open(data_yaml_path, "w") as f:
    f.write(data_yaml)

print("data.yaml updated.")

In [ ]:
#Train Model
from ultralytics import YOLO

model = YOLO("yolo26m.pt")

model.train(
    data="/content/ccpd26dark/data.yaml",

    #Core Settings
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.01,
    device=0,
    deterministic=True,

       #Auto Save Drive
    project="/content/drive/MyDrive",
    name="YOLOv26m_dark",
    save=True
)

In [ ]:
#Resume Model Training
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/YOLOv26m_dark3/weights/last.pt"
)

model.train(
    resume=True,
    project="/content/drive/MyDrive",
    name="YOLOv26m_dark3"
)

In [ ]:
#load best model change according model used RTDETR or YOLO
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/deeplearning_assignment_1/trained models/YOLOv26_normal/weights/best.pt")

In [ ]:
#Run Test Data
metrics = model.val(
    data="/content/ccpd26dark/data.yaml",
    split="test",
)

In [ ]:
#View Test Results
map50 = metrics.box.map50
map5095 = metrics.box.map

print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)

In [ ]:
#save test result
import pandas as pd
import os

save_dir = "/content/drive/MyDrive/deeplearning_assignment_1/YOLOv26_normal"
os.makedirs(save_dir, exist_ok=True)

df = pd.DataFrame([{
    "model": "YOLOv26",
    "condition": "normal",
    "mAP@0.5": metrics.box.map50,
    "mAP@0.5:0.95": metrics.box.map
}])

df.to_csv(f"{save_dir}/results.csv", index=False)